# AIQ Test Generation (A41)

Generates test Q&A pairs from pipeline detection results.

**Inputs (from pipeline_detection results):**
- Processed chunks (what to generate questions about)
- A30 findings (clarity issues → clarity probes)
- A31 findings (governance issues → governance probes)
- A32 findings (contradictions → consistency probes)
- Domain context (for natural question phrasing)

**Generates 4 categories:**
1. Topic coverage — can RAG find the right chunk?
2. Governance probes — is unsafe content blocked?
3. Clarity probes — are clarity fixes reflected?
4. Consistency probes — is the correct version served?

**Output:** test_set.csv with questions, ground truth, expected behavior

## Cell 1: Configuration

In [4]:
import sys, os, json, csv, random
if os.path.basename(os.getcwd()) == 'examples':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

RESULTS_FILE = '../results/neuroloft_kb_results.json'
if not os.path.exists(RESULTS_FILE): RESULTS_FILE = 'results/neuroloft_kb_results.json'

MAX_ROWS = 10
USE_LLM = True  # Recommended for quality questions
LLM_API_KEY = ''
for kp in ['../../../notebooks/doc/Key_o.txt', 'key.txt']:
    if os.path.exists(kp):
        with open(kp) as _kf: LLM_API_KEY = _kf.read().strip()
        break

# === TOPIC COVERAGE ===
# Options:
#   TOPIC_COVERAGE = 1.0    → 100% of chunks (one question per chunk)
#   TOPIC_COVERAGE = 0.5    → 50% of chunks (random sample)
#   TOPIC_COVERAGE = 10     → exactly 10 questions (integer = fixed count)
TOPIC_COVERAGE = 1.0

llm_call = None
if USE_LLM and LLM_API_KEY:
    from openai import OpenAI
    _client = OpenAI(api_key=LLM_API_KEY)
    def llm_call(prompt):
        return _client.chat.completions.create(model='gpt-4o-mini', messages=[{'role':'user','content':prompt}], max_tokens=300, temperature=0.0).choices[0].message.content.strip()

def show(items, fmt, label='items'):
    for item in items[:MAX_ROWS]: print(fmt(item))
    if len(items) > MAX_ROWS: print(f'  ... and {len(items)-MAX_ROWS} more {label}')

print(f'Results: {os.path.exists(RESULTS_FILE)} | LLM: {"YES" if llm_call else "NO"} | Topic coverage: {TOPIC_COVERAGE}')

## Cell 2: Load Detection Results

In [7]:
with open(RESULTS_FILE, encoding='utf-8') as f:
    results = json.load(f)

from aiq.core.types import Chunk, DomainContext

# Load chunks
chunks = [Chunk(chunk_id=c['chunk_id'], heading=c.get('heading',''), content=c['content'], words=c.get('words', len(c['content'].split())), source_type='text') for c in results['chunks']]

# Load domain context
dc = results.get('domain_context', {})
domain_ctx = DomainContext(
    domain_type=dc.get('domain_type', 'general'),
    company_name=dc.get('company_name', ''),
    actors=dc.get('actors', {}),
    acronyms=dc.get('acronyms', {}),
    product_names=dc.get('products', []),
)

# Load findings
findings = results.get('findings', {})
a30_findings = findings.get('a30', [])
a31_findings = findings.get('a31', [])
a32_findings = findings.get('a32', [])

print(f'Chunks: {len(chunks)}')
print(f'Domain: {domain_ctx.domain_type}')
print(f'Findings: A30={len(a30_findings)}, A31={len(a31_findings)}, A32={len(a32_findings)}')

if not findings:
    print('\nWARNING: No findings in results. Re-run pipeline_detection Cell 14 to save them.')

Chunks: 24
Domain: support
Findings: A30=28, A31=49, A32=2


## Cell 3: Generate Test Pairs (A41)

In [ ]:
# === TOPIC QUESTION GENERATION ===

QUESTION_PROMPT = """You are generating a retrieval test question for a knowledge base.

CONTENT:
{content}

TASK: Write ONE question that a person reading this knowledge base would naturally ask, where this content is the correct and complete answer.

RULES:
- The question must be answerable ONLY from this content
- Use natural language (how a real person would ask, not robotic)
- Be specific to the facts in the content (names, numbers, processes, dates)
- Do NOT ask "what is this about" or "can you help me with X"
- Do NOT mention section names, chunk IDs, or internal structure
- The question should have ONE clear answer, not be open-ended

Return ONLY the question, nothing else."""

ANSWER_PROMPT = """Given this question and source content, write a concise ground truth answer.
Include ONLY the key facts needed to answer the question.
Do not copy the entire content — extract the specific answer.

Question: {question}
Content: {content}

Answer (1-3 sentences max):"""

def get_clean_content(chunk):
    content = chunk.content
    if content.startswith(chunk.heading):
        parts = content.split('. ', 2)
        if len(parts) > 2:
            content = parts[2]
    return content

# Select chunks based on TOPIC_COVERAGE
content_chunks = [c for c in chunks if c.words > 0]

if isinstance(TOPIC_COVERAGE, float) and 0 < TOPIC_COVERAGE <= 1.0:
    count = max(1, int(len(content_chunks) * TOPIC_COVERAGE))
    if TOPIC_COVERAGE < 1.0:
        selected = random.sample(content_chunks, count)
    else:
        selected = content_chunks
elif isinstance(TOPIC_COVERAGE, int) and TOPIC_COVERAGE > 0:
    count = min(TOPIC_COVERAGE, len(content_chunks))
    selected = random.sample(content_chunks, count)
else:
    selected = content_chunks

print(f'Topic coverage: {len(selected)}/{len(content_chunks)} chunks ({len(selected)/len(content_chunks):.0%})')
print(f'Generating questions...
')

topic_pairs = []
for c in selected:
    content = get_clean_content(c)[:500]
    if llm_call:
        q = llm_call(QUESTION_PROMPT.format(content=content))
        a = llm_call(ANSWER_PROMPT.format(question=q, content=content))
    else:
        q = f'What information is available about {c.heading}?'
        sents = [s.strip() for s in content.split('.') if len(s.strip()) > 15]
        a = '. '.join(sents[:2]) + '.' if sents else content[:200]
    
    topic_pairs.append({
        'id': f'T{len(topic_pairs)+1:02d}',
        'category': 'topic',
        'question': q,
        'ground_truth': a,
        'expected_behavior': 'answer',
        'chunk_id': c.chunk_id,
        'heading': c.heading,
    })
    print(f'  [{c.chunk_id}] Q: {q}')
    print(f'          A: {a[:80]}...')
    print()

print(f'Generated: {len(topic_pairs)} topic questions')

### ✏️ Review Generated Pairs
Remove bad questions, fix ground truth, adjust expected behavior.

In [ ]:
# Remove a bad question:
# pairs = [p for p in pairs if p.pair_id != 'topic_3']

# Fix ground truth:
# for p in pairs:
#     if p.pair_id == 'gov_1':
#         p.expected_answer = 'No response founded'

# Add a custom question:
# from aiq.a41.generator import QAPair
# pairs.append(QAPair(pair_id='custom_1', question='What is the late fee?', expected_answer='1.5% monthly', expected_behavior='answer', source='user', reasoning='Custom test'))

print(f'{len(pairs)} pairs after review')

## Cell 4: Export Test Set

In [ ]:
# Save as CSV for use in pipeline_testing notebooks
out_path = 'test_set_generated.csv'
with open(out_path, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'category', 'question', 'ground_truth', 'expected_behavior', 'issue_ref', 'review_note'])
    writer.writeheader()
    for p in pairs:
        writer.writerow({
            'id': p.pair_id,
            'category': p.source,
            'question': p.question,
            'ground_truth': p.expected_answer,
            'expected_behavior': p.expected_behavior,
            'issue_ref': '',
            'review_note': f'Auto-generated ({p.reasoning[:50]})',
        })

print(f'Saved: {out_path} ({len(pairs)} pairs)')
print(f'\nUse this in pipeline_testing notebooks:')
print(f'  TEST_SET_FILE = \'test_set_generated.csv\'')

# Also save to results folder
results_dir = os.path.join(os.path.dirname(os.getcwd()), 'results') if os.path.basename(os.getcwd()) == 'examples' else os.path.join(os.getcwd(), 'results')
os.makedirs(results_dir, exist_ok=True)
doc_name = os.path.splitext(os.path.basename(results['input_file']))[0]
json_path = os.path.join(results_dir, f'{doc_name}_test_set.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump([{'id': p.pair_id, 'category': p.source, 'question': p.question, 'ground_truth': p.expected_answer, 'expected_behavior': p.expected_behavior, 'reasoning': p.reasoning} for p in pairs], f, indent=2, ensure_ascii=False)
print(f'Also saved: {json_path}')